In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import sys
import numpy as np
import time
import matplotlib.pyplot as plt
import subprocess
import platform


build_dir = "build"
configType = "Release" #"Debug" Release
#pbindDir = r"c:\Users\CathalMcLoughlin\anaconda3\envs\morpho311\Lib\site-packages\pybind11\share\cmake\pybind11"

# Ensure build directory exists
os.makedirs(build_dir, exist_ok=True)

#automatically identify the pybind11 directory
pbindDir = subprocess.run(
    ["python", "-c", "import pybind11; print(pybind11.get_cmake_dir())"],
    capture_output=True, text=True).stdout.strip()

cmakeArgs = [
    'cmake', '-S', ".", '-B', build_dir,
    f"-Dpybind11_DIR={pbindDir}",
    f"-DCMAKE_BUILD_TYPE={configType}"
]

# Portable Fix: If on Mac, point to the Homebrew LLVM compilers
if platform.system() == "Darwin":
    print("on mac")
    # Homebrew usually installs llvm here
    llvm_prefix = "/opt/homebrew/opt/llvm"
    if os.path.exists(llvm_prefix):
        cmakeArgs.append(f"-DCMAKE_C_COMPILER={llvm_prefix}/bin/clang")
        cmakeArgs.append(f"-DCMAKE_CXX_COMPILER={llvm_prefix}/bin/clang++")
        # Help FindOpenMP find the library
        cmakeArgs.append(f"-DOpenMP_ROOT={llvm_prefix}")



# Run CMake Configuration Step
pybind11_path = subprocess.run(
    ["python", "-c", "import pybind11; print(pybind11.get_cmake_dir())"],
    capture_output=True, text=True).stdout.strip()


out1 = subprocess.run(cmakeArgs,
 stdout=subprocess.PIPE, stderr=subprocess.PIPE)

print(f"Out1 (CMake Configure): {out1.stdout.decode('utf-8')}")
print(f"Err1 (CMake Configure): {out1.stderr.decode('utf-8')}")

# If configuration fails, exit early
if out1.returncode != 0:
    print("CMake configuration failed. Exiting.")
    exit(1)

# Run CMake Build Step
out2 = subprocess.run(['cmake', '--build', build_dir, '--config', configType], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print(f"Out2 (CMake Build): {out2.stdout.decode('utf-8')}")
print(f"Err2 (CMake Build): {out2.stderr.decode('utf-8')}")

# If build fails, print error
if out2.returncode != 0:
    print("CMake build failed.")
    assert False

Out1 (CMake Configure): -- Selecting Windows SDK version 10.0.18362.0 to target Windows 6.2.9200.
-- Found pybind11: C:/Users/catha/anaconda3/envs/morpho25/Lib/site-packages/pybind11/include (found version "2.13.6")
-- Configuring done
-- Generating done
-- Build files have been written to: C:/Users/catha/source/repos/pythonCPPtesting.ipynb/build

Err1 (CMake Configure): 
Out2 (CMake Build): Microsoft (R) Build Engine version 16.6.0+5ff7b0c9e for .NET Framework
Copyright (C) Microsoft Corporation. All rights reserved.

cl : command line warning D9035: option 'o' has been deprecated and will be removed in a future release [C:\Users\catha\source\repos\pythonCPPtesting.ipynb\build\cathalTest.vcxproj]
  functionDefs.cpp
cl : command line warning D9002: ignoring unknown option '-O0' [C:\Users\catha\source\repos\pythonCPPtesting.ipynb\build\cathalTest.vcxproj]
  myPBmodule.cpp
     Creating library C:/Users/catha/source/repos/pythonCPPtesting.ipynb/build/Release/cathalTest.lib and object C:/

In [2]:
moduleDir = os.path.join(build_dir, configType)
if moduleDir not in sys.path:
    sys.path.append(moduleDir)

import cathalTest

print(dir(cathalTest))  

['__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'ompMaxThreads', 'paritySum', 'solveNeuralSystem', 'solveNeuralSystemSingleThread', 'testFunc']


In [3]:
cathalTest.testFunc(np.array([1,1,2],dtype=int),np.array([0.0,1.1,2.1],dtype=np.float64))

In [4]:
cathalTest.paritySum(20)

10

In [6]:
cathalTest.ompMaxThreads()

10

In [4]:
myDict = {"neuronId": np.arange(3),
          "voltages":np.random.random(5),}

cathalTest.solveNeuralSystem(myDict,2,0.1)

TypeError: solveNeuralSystem(): incompatible function arguments. The following argument types are supported:
    1. (myDict: dict, recordingInfo: dict, timeStep: typing.SupportsFloat, nIterations: typing.SupportsInt, samplingInterval: typing.SupportsInt) -> object

Invoked with: {'neuronId': array([0, 1, 2]), 'voltages': array([0.45852292, 0.19764658, 0.21065638, 0.71810505, 0.94416906])}, 2, 0.1